# UAS Machine Learning
## Prediksi Kondisi Sosial Provinsi Jawa Barat

**Nama:** Muhammad Rifqy Saputra  
**NIM:** 2307046  
**Kelas:** D4SIKC.3B  
**Pilar Smart City:** Smart Living / Smart Governance

Notebook ini berisi dataset, EDA sederhana, preprocessing, training minimal dua algoritma, evaluasi, prediksi, dan interpretasi Smart City.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVR, SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, mean_absolute_error, mean_squared_error, r2_score
import joblib, json

FEATURES = ['gini_ratio','tingkat_penganggur_terbuka','rata_rata_inflasi_tahunan','indeks_pembangunan_manusia']
TARGET = 'kemiskinan_tahun_depan'
HIST = 'persentase_penduduk_miskin'

def smart_number(v):
    if pd.isna(v): return np.nan
    if isinstance(v, (pd.Timestamp, datetime)):
        return float(v.day + v.month/100)
    s = str(v).replace('%','').replace(',','.').strip()
    try: x = float(s)
    except Exception: return np.nan
    if abs(x) > 1000:
        while abs(x) > 20: x /= 10
    return float(x)

df_raw = pd.read_excel('dataset_kemiskinan_jawa_barat.xlsx')
df = df_raw.copy()
for col in [HIST, TARGET, 'gini_ratio','tingkat_penganggur_terbuka','rata_rata_inflasi_tahunan','indeks_pembangunan_manusia']:
    df[col] = df[col].apply(smart_number)
df['gini_ratio'] = df['gini_ratio'].apply(lambda x: np.nan if pd.isna(x) or x == 0 else (x/1000 if x > 1 else x))
df['tahun'] = pd.to_numeric(df['tahun'], errors='coerce')
print('Raw rows:', len(df_raw))
print('Clean rows:', len(df))
df.head()

## EDA Sederhana
EDA dilakukan untuk melihat jumlah data, periode tahun, missing value, dan tren indikator sosial ekonomi.

In [ ]:
print('Shape:', df.shape)
print('Tahun:', int(df['tahun'].min()), '-', int(df['tahun'].max()))
print('Missing value utama:')
print(df[FEATURES + [TARGET]].isna().sum())

trend = df.groupby('tahun').agg(
    rata_rata_kemiskinan=(HIST, 'mean'),
    rata_rata_target=(TARGET, 'mean'),
    rata_rata_gini=('gini_ratio','mean'),
    rata_rata_tpt=('tingkat_penganggur_terbuka','mean'),
    rata_rata_ipm=('indeks_pembangunan_manusia','mean')
).reset_index()
trend

## Preprocessing dan Split Data
Perbaikan penting pada UAS ini adalah cleaning nilai numerik, scaling gini ratio/inflasi yang salah format, dan deduplikasi fitur-target sebelum train-test split agar hasil evaluasi tidak 1.0000 semua.

In [ ]:
df_model = df.dropna(subset=[TARGET]).drop_duplicates(subset=FEATURES + [TARGET]).copy()
X = df_model[FEATURES]
y_reg = df_model[TARGET].astype(float)
low_threshold = float(y_reg.quantile(0.33))
high_threshold = float(y_reg.quantile(0.66))

def priority(v):
    if v <= low_threshold: return 'Low Priority'
    if v <= high_threshold: return 'Medium Priority'
    return 'High Priority'

y_cls = y_reg.apply(priority)
X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.30, random_state=7, stratify=y_cls
)
print('Modeling rows setelah deduplikasi:', len(df_model))
print('Train:', len(X_train), '| Test:', len(X_test))
print('Threshold:', low_threshold, high_threshold)
y_cls.value_counts()

In [ ]:
regression_candidates = {
    'RandomForestRegressor': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', RandomForestRegressor(n_estimators=120, max_depth=3, random_state=42, n_jobs=-1))]),
    'SVR': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', SVR(C=1.0, epsilon=0.05))]),
    'Polynomial Regression': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('poly', PolynomialFeatures(degree=2, include_bias=False)), ('model', LinearRegression())])
}
reg_rows = []
for name, model in regression_candidates.items():
    model.fit(X_train, y_reg_train)
    pred = model.predict(X_test)
    reg_rows.append({'model': name, 'r2_score': r2_score(y_reg_test, pred), 'mae': mean_absolute_error(y_reg_test, pred), 'rmse': np.sqrt(mean_squared_error(y_reg_test, pred))})
regression_metrics = pd.DataFrame(reg_rows).sort_values('r2_score', ascending=False).reset_index(drop=True)
regression_metrics

In [ ]:
classification_candidates = {
    'RandomForestClassifier': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', RandomForestClassifier(n_estimators=120, max_depth=2, random_state=42, n_jobs=-1))]),
    'SVC': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', SVC(C=1.0))]),
    'LogisticRegression': Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])
}
cls_rows = []
cls_preds = {}
for name, model in classification_candidates.items():
    model.fit(X_train, y_cls_train)
    pred = model.predict(X_test)
    cls_preds[name] = pred
    cls_rows.append({'model': name, 'accuracy': accuracy_score(y_cls_test, pred), 'precision_weighted': precision_score(y_cls_test, pred, average='weighted', zero_division=0), 'recall_weighted': recall_score(y_cls_test, pred, average='weighted', zero_division=0), 'f1_weighted': f1_score(y_cls_test, pred, average='weighted', zero_division=0)})
classification_metrics = pd.DataFrame(cls_rows).sort_values('accuracy', ascending=False).reset_index(drop=True)
classification_metrics

In [ ]:
best_cls = classification_metrics.loc[0, 'model']
labels = ['Low Priority', 'Medium Priority', 'High Priority']
print('Best classification model:', best_cls)
print('Confusion matrix:')
print(pd.DataFrame(confusion_matrix(y_cls_test, cls_preds[best_cls], labels=labels), index=labels, columns=labels))
print('\nClassification report:')
print(classification_report(y_cls_test, cls_preds[best_cls], zero_division=0))

## Perbandingan Model dan Alasan Pemilihan
Model regresi terbaik dipilih berdasarkan R2 tertinggi dengan MAE dan RMSE yang rendah. Model klasifikasi terbaik dipilih berdasarkan accuracy serta F1-score weighted. Random Forest digunakan karena mampu menangani hubungan non-linear antar indikator sosial ekonomi dan tetap mudah dijelaskan melalui feature importance.

In [ ]:
import pandas as pd
sample = pd.DataFrame([{'tahun': 2026, 'gini_ratio': 0.42, 'tingkat_penganggur_terbuka': 6.7, 'rata_rata_inflasi_tahunan': 3.1, 'indeks_pembangunan_manusia': 73.5}])
final_reg = joblib.load('ml_artifacts/poverty_model_bundle.pkl')['regression_model']
pred = float(final_reg.predict(sample[FEATURES])[0])
print('Contoh input:')
print(sample)
print('Prediksi kemiskinan:', round(pred, 4))
print('Prioritas:', priority(pred))

## Analisis Smart City

1. **Bagaimana model membantu Smart City?** Model membantu pemerintah melihat estimasi risiko sosial dan menentukan prioritas intervensi berbasis data.
2. **Manfaat bagi pemerintah/masyarakat:** pemerintah dapat menyusun program bantuan, pelatihan kerja, dan penguatan UMKM dengan lebih tepat sasaran; masyarakat memperoleh respons kebijakan yang lebih cepat.
3. **Risiko jika prediksi salah:** bantuan bisa tidak tepat sasaran atau wilayah rentan terlambat ditangani, sehingga output model wajib dipakai sebagai pendukung keputusan dan divalidasi di lapangan.
4. **Privasi data:** dataset yang digunakan bersifat agregat, bukan data pribadi individu. Jika dikembangkan memakai data warga, identitas perlu dianonimkan dan akses sistem harus dibatasi.